[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BSLJunhyeonJeon/AI_COP/blob/main/session2/notebooks/02_inference_demo.ipynb)

# session2 · 02 · 실습1 · 사전학습 모델 추론 데모 (혈액세포)

- **이 노트북에서 배우는 것**: 분류·감지·분할 대표 사전학습 모델이 **일반 사진엔 성공, 혈액 이미지엔 실패**하는 대비를 본다.
- **입력**: 없음 — 성공용 일반 이미지 1장 + 실패용 혈액 이미지를 코드로 확보.
- **출력**: 태스크별 성공/실패 그림 + 종합 그림 (`outputs/`)

> **핵심**: 일반 도메인(ImageNet/COCO/ADE20K)으로 학습된 모델은 혈액세포를 배운 적이 없어 빗나간다. 이 실패가 **다음 세션 전이학습의 동기**다.
> 학습/파인튜닝 없음(추론만). GPU 있으면 사용, 없으면 CPU(모델이 작아 가능). 모델: ImageNet 분류기 · YOLO11n · SegFormer-b0.
> 그림 안 글자는 폰트 호환을 위해 영문(설명은 한국어 출력/마크다운).

In [ ]:
# 셀 1 · 환경 감지 + 프로젝트 루트 확보 (00/01 과 동일 패턴 — 분기는 이 셀 한 곳)
import os, subprocess

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

SESSION = "session2"
REPO_URL = "https://github.com/BSLJunhyeonJeon/AI_COP"
REPO_DIR = "/content/AI_COP"
SESSION_DIR = REPO_DIR + "/" + SESSION


def acquire_project():
    """코랩: 레포를 /content/AI_COP 로 클론 후 session2 를 PROJECT_ROOT 로 반환(실패 시 None)."""
    if os.path.isdir(REPO_DIR):
        print("이미 존재:", REPO_DIR, "(재클론 건너뜀)")
        try:
            r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
            if r.returncode != 0:
                print("  (git pull 실패 — 기존 캐시 버전 사용)")
        except Exception as e:
            print("  (git pull 건너뜀:", e, ")")
    else:
        print("레포 클론:", REPO_URL, "->", REPO_DIR)
        try:
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
        except Exception as e:
            print("clone 실패(네트워크/권한 확인):", e)
    return SESSION_DIR if os.path.isdir(SESSION_DIR) else None


def find_root_local(marker="requirements.txt"):
    """로컬: requirements.txt 를 마커로 상위 탐색해 session2 루트를 찾는다."""
    start = os.path.abspath(os.getcwd())
    d = start
    while True:
        if os.path.exists(os.path.join(d, marker)):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            print("[주의] '" + marker + "' 를 못 찾음. 현재 폴더를 루트로 가정:", start)
            return start
        d = parent


PROJECT_ROOT = acquire_project() if IN_COLAB else find_root_local()
if not (PROJECT_ROOT and os.path.isdir(PROJECT_ROOT)):
    raise RuntimeError(
        "세션 루트를 확보하지 못했습니다. "
        "코랩이면 레포 클론 실패이니 네트워크 확인 후 이 셀(셀 1)을 다시 ▶ 실행하세요. "
        "로컬이면 session2/ 안에서 노트북을 열었는지 확인하세요."
    )
os.chdir(PROJECT_ROOT)
import torch
print("실행 환경   :", "Colab" if IN_COLAB else "Local")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("장치(device):", "GPU(cuda)" if torch.cuda.is_available() else "CPU")

In [ ]:
# 셀 2 · 의존성 설치 (torchvision/ultralytics/transformers). 코랩 사전설치본·torch 는 재설치 안 함.
import os, sys, subprocess

if not os.path.exists("requirements.txt"):
    raise RuntimeError("requirements.txt 를 찾지 못했습니다. 셀 1을 먼저 ▶ 실행하세요.")
print("requirements.txt 설치 중... (torchvision/ultralytics/transformers 포함, 수십 초 걸릴 수 있음)")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
if r.returncode != 0:
    raise RuntimeError("pip install 실패. 위 로그 확인 후 이 셀(셀 2)을 다시 ▶ 실행하세요.")

print("\n[설치된 새 의존성 버전 — requirements.txt 핀에 사용]")
for mod in ["torch", "torchvision", "ultralytics", "transformers"]:
    try:
        m = __import__(mod)
        print("  -", mod, ":", getattr(m, "__version__", "?"))
    except Exception as e:
        print("  -", mod, ": import 실패 (", e, ")")

# ultralytics 등이 torch 를 덮어쓰지 않았는지 확인(스펙: 기존 torch 유지)
import torch as _t
if not _t.__version__.startswith("2.11"):
    print("\n[경고] torch 가", _t.__version__, "로 바뀌었습니다(2.11 예상).")
    print("       ultralytics 등이 torch 를 덮어썼을 수 있습니다 — 런타임 재시작 후 다시 실행하거나 핀을 점검하세요.")

In [ ]:
# 셀 3 · 입력 확보: 성공용 일반 이미지 1장 + 실패용 혈액 이미지(세포 1장, 도말 1장)
# (1단계 데이터가 있으면 재사용, 없으면 직접 확보 — 노트북 단독 실행 보장)
import os, glob, urllib.request
from PIL import Image

demo = os.path.join("data", "_demo_inputs")
os.makedirs(demo, exist_ok=True)
SUCCESS = os.path.join(demo, "success.jpg")
BLOOD_CELL = os.path.join(demo, "blood_cell.png")
BLOOD_SMEAR = os.path.join(demo, "blood_smear.jpg")

# (1) 성공용: 안정적인 공개 일반 장면(거리/버스/사람) — 세 모델의 제 도메인
if not os.path.exists(SUCCESS):
    urls = ["https://ultralytics.com/images/bus.jpg",
            "https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/assets/bus.jpg",
            "https://github.com/pytorch/hub/raw/master/images/dog.jpg"]  # 마지막 폴백(다른 호스트)
    for u in urls:
        try:
            urllib.request.urlretrieve(u, SUCCESS)
            print("성공용 일반 이미지 받음:", u)
            break
        except Exception as e:
            print("  (다운로드 실패:", u, "->", e, ")")
    if not os.path.exists(SUCCESS):
        print("[주의] 성공용 이미지 확보 실패 — 네트워크 확인 후 셀 3 재실행")

# (2) 실패용 혈액세포(분류/분할): 1단계 데이터 재사용, 없으면 medmnist 1장
if not os.path.exists(BLOOD_CELL):
    ex = glob.glob(os.path.join("data", "classification", "*", "*.png"))
    if ex:
        Image.open(ex[0]).convert("RGB").save(BLOOD_CELL)
        print("혈액세포 이미지(1단계 재사용):", ex[0])
    else:
        try:
            from medmnist import BloodMNIST
            try:
                ds = BloodMNIST(split="train", download=True, size=64)
            except TypeError:
                ds = BloodMNIST(split="train", download=True)
            Image.fromarray(ds.imgs[0]).convert("RGB").save(BLOOD_CELL)
            print("혈액세포 이미지(medmnist 확보)")
        except Exception as e:
            print("[주의] 혈액세포 이미지 확보 실패:", e)

# (3) 실패용 혈액 도말(감지): 1단계 데이터 재사용, 없으면 BCCD 1장
if not os.path.exists(BLOOD_SMEAR):
    ex = sorted(glob.glob(os.path.join("data", "detection", "*.jpg")))
    if ex:
        Image.open(ex[0]).convert("RGB").save(BLOOD_SMEAR)
        print("혈액 도말 이미지(1단계 재사용):", ex[0])
    else:
        src = os.path.join("data", "_bccd_src")
        if not os.path.isdir(src):
            subprocess = __import__("subprocess")
            subprocess.run(["git", "clone", "--depth", "1",
                            "https://github.com/Shenggan/BCCD_Dataset", src], check=False)
        bccd = sorted(glob.glob(os.path.join(src, "BCCD", "JPEGImages", "*.jpg")))
        if bccd:
            Image.open(bccd[0]).convert("RGB").save(BLOOD_SMEAR)
            print("혈액 도말 이미지(BCCD 확보)")
        else:
            print("[주의] 혈액 도말 이미지 확보 실패(네트워크). 셀 3 재실행")

ready = {"success": os.path.exists(SUCCESS), "blood_cell": os.path.exists(BLOOD_CELL),
         "blood_smear": os.path.exists(BLOOD_SMEAR)}
print("입력 준비 상태:", ready)
if not all(ready.values()):
    print("[주의] 일부 입력이 없습니다. 위 메시지를 보고 네트워크 확인 후 셀 3을 다시 ▶ 실행하세요.")

In [ ]:
# 셀 4 · 분류 데모 (torchvision ImageNet 분류기, 1000 일반 클래스)
# 성공(일반 이미지)=합리적 라벨 / 실패(혈액세포)=엉뚱한 라벨 (혈액세포 클래스가 1000개에 없음)
import os, torch
import matplotlib.pyplot as plt
from PIL import Image
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

device = "cuda" if torch.cuda.is_available() else "cpu"
weights = MobileNet_V3_Small_Weights.IMAGENET1K_V1
clf = mobilenet_v3_small(weights=weights).eval().to(device)
preprocess = weights.transforms()
categories = weights.meta["categories"]


def top3(path):
    img = Image.open(path).convert("RGB")
    x = preprocess(img).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = clf(x).softmax(1)[0]
    p, idx = probs.topk(3)
    return img, [(categories[int(i)], float(s)) for s, i in zip(p, idx)]


demo = os.path.join("data", "_demo_inputs")
items = [("SUCCESS: street photo", os.path.join(demo, "success.jpg")),
         ("FAIL: blood cell", os.path.join(demo, "blood_cell.png"))]
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
for col, (tag, path) in enumerate(items):
    if os.path.exists(path):
        img, preds = top3(path)
        ax[col].imshow(img)
        ax[col].set_title(tag)
        ax[col].set_xlabel("\n".join("%s (%.2f)" % (n, s) for n, s in preds))
        print(tag, "-> top1:", preds[0][0])
    else:
        ax[col].set_title(tag + " (입력 없음 — 셀 3)")
    ax[col].set_xticks([]); ax[col].set_yticks([])
fig.suptitle("Classification (ImageNet 1000): general OK, blood cell -> wrong label")
os.makedirs("outputs", exist_ok=True)
plt.tight_layout()
plt.savefig(os.path.join("outputs", "02_classification.png"), dpi=130, bbox_inches="tight")
plt.show()
print("저장: outputs/02_classification.png  (혈액세포는 1000개 일반 클래스에 없어 엉뚱하게 분류됨)")

In [ ]:
# 셀 5 · 감지 데모 (Ultralytics YOLO11n, COCO 80클래스)
# 성공=사람/버스 등 박스 / 실패(혈액 도말)=박스 거의 없음·엉뚱 (COCO에 혈액세포 없음)
import os, torch
import matplotlib.pyplot as plt
from ultralytics import YOLO

device = "cuda" if torch.cuda.is_available() else "cpu"
det_model = YOLO("yolo11n.pt")          # COCO 사전학습, 자동 다운로드(~5MB)
det_model.to(device)                    # 셀 4/6 과 동일하게 명시적 디바이스
demo = os.path.join("data", "_demo_inputs")
items = [("SUCCESS: street photo", os.path.join(demo, "success.jpg")),
         ("FAIL: blood smear", os.path.join(demo, "blood_smear.jpg"))]
fig, ax = plt.subplots(1, 2, figsize=(11, 5))
for col, (tag, path) in enumerate(items):
    if os.path.exists(path):
        res = det_model(path, verbose=False, device=device)[0]
        plotted = res.plot()                 # 박스가 그려진 BGR ndarray
        ax[col].imshow(plotted[:, :, ::-1])  # BGR -> RGB
        n = len(res.boxes)
        ax[col].set_title(tag + "  (boxes: %d)" % n)
        print(tag, "-> 박스", n, "개")
    else:
        ax[col].set_title(tag + " (입력 없음 — 셀 3)")
    ax[col].axis("off")
fig.suptitle("Detection (YOLO11n / COCO 80): general OK, blood smear -> few/none (no blood-cell class)")
os.makedirs("outputs", exist_ok=True)
plt.tight_layout()
plt.savefig(os.path.join("outputs", "02_detection.png"), dpi=130, bbox_inches="tight")
plt.show()
print("저장: outputs/02_detection.png  (COCO 80클래스에 혈액세포가 없어 도말에선 박스가 거의 없음)")

In [ ]:
# 셀 6 · 분할 데모 (HuggingFace SegFormer-b0, ADE20K 150 시맨틱 클래스)
# 성공=도로/사람/차 등 의미있는 분할 / 실패(혈액)=엉뚱한 ADE 클래스로 칠함
import os, numpy as np, torch
import matplotlib.pyplot as plt
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForSemanticSegmentation

device = "cuda" if torch.cuda.is_available() else "cpu"
# SegFormer-b0 (~15MB)는 첫 실행 시 HuggingFace 에서 자동 다운로드됩니다(수십 초).
name = "nvidia/segformer-b0-finetuned-ade-512-512"
processor = AutoImageProcessor.from_pretrained(name)
seg_model = AutoModelForSemanticSegmentation.from_pretrained(name).eval().to(device)

rng = np.random.default_rng(0)
palette = rng.integers(0, 255, size=(256, 3), dtype=np.uint8)   # 클래스별 색(시드 고정 — 재현성)


def segment(path):
    img = Image.open(path).convert("RGB")
    inputs = processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = seg_model(**inputs).logits                     # (1, 150, h/4, w/4)
    up = torch.nn.functional.interpolate(logits, size=img.size[::-1], mode="bilinear", align_corners=False)
    seg = up.argmax(1)[0].cpu().numpy()
    return img, seg


demo = os.path.join("data", "_demo_inputs")
items = [("SUCCESS: street photo", os.path.join(demo, "success.jpg")),
         ("FAIL: blood cell", os.path.join(demo, "blood_cell.png"))]
fig, ax = plt.subplots(1, 2, figsize=(11, 5))
for col, (tag, path) in enumerate(items):
    if os.path.exists(path):
        img, seg = segment(path)
        overlay = (0.5 * np.array(img) + 0.5 * palette[seg % 256]).astype(np.uint8)
        ax[col].imshow(overlay)
        ax[col].set_title(tag + "  (ADE classes: %d)" % len(np.unique(seg)))
        print(tag, "-> ADE 클래스", len(np.unique(seg)), "종")
    else:
        ax[col].set_title(tag + " (입력 없음 — 셀 3)")
    ax[col].axis("off")
fig.suptitle("Segmentation (SegFormer-b0 / ADE20K 150): general meaningful, blood -> wrong ADE classes")
os.makedirs("outputs", exist_ok=True)
plt.tight_layout()
plt.savefig(os.path.join("outputs", "02_segmentation.png"), dpi=130, bbox_inches="tight")
plt.show()
print("저장: outputs/02_segmentation.png  (혈액 이미지를 도로/벽 등 엉뚱한 ADE 클래스로 칠함)")

In [ ]:
# 셀 7 · 종합 + 다음 세션으로의 다리 (검증 포함)
import os
import matplotlib.pyplot as plt
from PIL import Image

panels = [("Classification", "outputs/02_classification.png"),
          ("Detection", "outputs/02_detection.png"),
          ("Segmentation", "outputs/02_segmentation.png")]
have = [(t, p) for t, p in panels if os.path.exists(p)]
if have:
    fig, ax = plt.subplots(len(have), 1, figsize=(11, 4.4 * len(have)))
    if len(have) == 1:
        ax = [ax]
    for a, (t, p) in zip(ax, have):
        a.imshow(Image.open(p)); a.set_title(t); a.axis("off")
    fig.suptitle("Pretrained general-domain models: OK on general photos, miss on blood -> motivates transfer learning")
    plt.tight_layout()
    plt.savefig("outputs/02_summary.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("저장: outputs/02_summary.png")

print("=" * 56)
print(" 2단계 · 추론 데모 검증/요약")
print("=" * 56)
for t, p in panels:
    print(" -", t.ljust(14), ":", "있음" if os.path.exists(p) else "없음", " (", p, ")")
if not all(os.path.exists(p) for _, p in panels):
    print(" [주의] 일부 결과 그림이 없습니다 — 셀 3(입력)부터 6까지 순서대로 다시 ▶ 실행하세요.")
print("=" * 56)
print("정리:")
print(" 세 모델(ImageNet 분류기 · YOLO11 · SegFormer) 모두 일반 사진엔 OK,")
print(" 혈액 이미지엔 빗나간다 — '너희 도메인'을 배운 적이 없어서.")
print(" => 다음 세션: 바로 이 모델들(YOLO11 · SegFormer)을 '너희 라벨'로 전이학습하면 맞히게 된다.")